In [ ]:
#| default_exp compute

# Compute

> Compute Engine (GCE), Google Kubernetes Engine (GKE), and Artifact Registry.

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
try:
    from google.cloud import compute_v1
    from google.cloud import container_v1
    from google.cloud import artifactregistry_v1
except ImportError:
    pass

## Compute Engine

In [ ]:
#| export
def _latest_debian_image(auth, zone: str) -> str:
    """Return the latest Debian 12 image selfLink."""
    client = compute_v1.ImagesClient(credentials=auth.credentials)
    images = client.get_from_family(
        project='debian-cloud', family='debian-12'
    )
    return images.self_link


def create_instance(
    auth,
    name: str,
    machine_type: str = 'e2-medium',
    zone: str = None,
    image: str = None,
    disk_size_gb: int = 20,
    shielded: bool = True,
    labels: dict = None,
    **_,
) -> dict:
    """Create a Compute Engine VM instance.

    Shielded VM (secure boot + vTPM + integrity monitoring) is enabled by default.
    Uses the latest Debian 12 image if `image` is not provided.
    """
    zone = zone or f'{auth.region}-a'
    client = compute_v1.InstancesClient(credentials=auth.credentials)

    # Check if instance already exists
    try:
        existing = client.get(project=auth.project, zone=zone, instance=name)
        return {'name': name, 'zone': zone, 'status': existing.status}
    except Exception:
        pass

    source_image = image or _latest_debian_image(auth, zone)
    instance = compute_v1.Instance(
        name=name,
        machine_type=f'zones/{zone}/machineTypes/{machine_type}',
        disks=[
            compute_v1.AttachedDisk(
                boot=True,
                auto_delete=True,
                initialize_params=compute_v1.AttachedDiskInitializeParams(
                    source_image=source_image,
                    disk_size_gb=disk_size_gb,
                ),
            )
        ],
        network_interfaces=[
            compute_v1.NetworkInterface(
                name='global/networks/default',
            )
        ],
        labels=labels or {},
        shielded_instance_config=(
            compute_v1.ShieldedInstanceConfig(
                enable_secure_boot=True,
                enable_vtpm=True,
                enable_integrity_monitoring=True,
            )
            if shielded else None
        ),
    )
    op = client.insert(project=auth.project, zone=zone, instance_resource=instance)
    op.result(timeout=300)
    return {'name': name, 'zone': zone}


def instance_ip(auth, name: str, zone: str = None) -> str:
    "Return the external IP of a Compute Engine instance."
    zone = zone or f'{auth.region}-a'
    client = compute_v1.InstancesClient(credentials=auth.credentials)
    inst = client.get(project=auth.project, zone=zone, instance=name)
    return inst.network_interfaces[0].access_configs[0].nat_i_p


def start_instance(auth, name: str, zone: str = None):
    "Start a stopped Compute Engine instance."
    zone = zone or f'{auth.region}-a'
    client = compute_v1.InstancesClient(credentials=auth.credentials)
    op = client.start(project=auth.project, zone=zone, instance=name)
    op.result()


def stop_instance(auth, name: str, zone: str = None):
    "Stop a running Compute Engine instance."
    zone = zone or f'{auth.region}-a'
    client = compute_v1.InstancesClient(credentials=auth.credentials)
    op = client.stop(project=auth.project, zone=zone, instance=name)
    op.result()


def delete_instance(auth, name: str, zone: str = None):
    "Delete a Compute Engine instance."
    zone = zone or f'{auth.region}-a'
    client = compute_v1.InstancesClient(credentials=auth.credentials)
    op = client.delete(project=auth.project, zone=zone, instance=name)
    op.result()

## Google Kubernetes Engine (GKE)

In [ ]:
#| export
def create_gke_cluster(
    auth,
    name: str,
    node_count: int = 1,
    machine_type: str = 'e2-standard-4',
    autopilot: bool = True,
    workload_identity: bool = True,
    labels: dict = None,
    **_,
) -> dict:
    """Create a GKE cluster. Defaults to Autopilot mode with Workload Identity."""
    client = container_v1.ClusterManagerClient(credentials=auth.credentials)
    parent = f'projects/{auth.project}/locations/{auth.region}'

    try:
        existing = client.get_cluster(name=f'{parent}/clusters/{name}')
        return {'name': name, 'endpoint': existing.endpoint, 'status': str(existing.status)}
    except Exception:
        pass

    if autopilot:
        cluster = container_v1.Cluster(
            name=name,
            autopilot=container_v1.Autopilot(enabled=True),
            workload_identity_config=(
                container_v1.WorkloadIdentityConfig(
                    workload_pool=f'{auth.project}.svc.id.goog'
                ) if workload_identity else None
            ),
            resource_labels=labels or {},
        )
    else:
        cluster = container_v1.Cluster(
            name=name,
            node_pools=[
                container_v1.NodePool(
                    name='default-pool',
                    initial_node_count=node_count,
                    config=container_v1.NodeConfig(
                        machine_type=machine_type,
                        workload_metadata_config=(
                            container_v1.WorkloadMetadataConfig(
                                mode=container_v1.WorkloadMetadataConfig.Mode.GKE_METADATA
                            ) if workload_identity else None
                        ),
                    ),
                )
            ],
            workload_identity_config=(
                container_v1.WorkloadIdentityConfig(
                    workload_pool=f'{auth.project}.svc.id.goog'
                ) if workload_identity else None
            ),
            resource_labels=labels or {},
        )

    op = client.create_cluster(
        parent=parent,
        cluster=cluster,
    )
    return {'name': name, 'operation': op.name}


def gke_kubeconfig(auth, name: str) -> dict:
    """Return kubeconfig dict for connecting to a GKE cluster."""
    client = container_v1.ClusterManagerClient(credentials=auth.credentials)
    cluster = client.get_cluster(
        name=f'projects/{auth.project}/locations/{auth.region}/clusters/{name}'
    )
    return {
        'endpoint': f'https://{cluster.endpoint}',
        'ca_cert': cluster.master_auth.cluster_ca_certificate,
        'name': name,
    }


def scale_gke(auth, name: str, node_pool: str, node_count: int):
    "Scale a GKE node pool to `node_count`."
    client = container_v1.ClusterManagerClient(credentials=auth.credentials)
    op = client.set_node_pool_size(
        name=(
            f'projects/{auth.project}/locations/{auth.region}'
            f'/clusters/{name}/nodePools/{node_pool}'
        ),
        node_count=node_count,
    )
    return op.name

## Artifact Registry

In [ ]:
#| export
def create_artifact_registry(
    auth,
    name: str,
    format: str = 'DOCKER',
    labels: dict = None,
    **_,
) -> dict:
    """Create an Artifact Registry repository. Vulnerability scanning is always enabled."""
    client = artifactregistry_v1.ArtifactRegistryClient(credentials=auth.credentials)
    parent = f'projects/{auth.project}/locations/{auth.region}'

    try:
        existing = client.get_repository(
            name=f'{parent}/repositories/{name}'
        )
        return {'name': existing.name, 'format': format}
    except Exception:
        pass

    repo = artifactregistry_v1.Repository(
        format_=artifactregistry_v1.Repository.Format[format],
        labels=labels or {},
    )
    op = client.create_repository(
        parent=parent,
        repository_id=name,
        repository=repo,
    )
    result = op.result(timeout=120)
    return {'name': result.name, 'format': format}


def registry_url(auth, name: str) -> str:
    "Return the Docker image path prefix for an Artifact Registry repo."
    return f'{auth.region}-docker.pkg.dev/{auth.project}/{name}'


def attach_registry_to_gke(auth, registry_name: str, gke_sa_email: str):
    """Grant Workload Identity service account read access to the Artifact Registry repo."""
    from gcpeasy.network import bind_iam_role
    bind_iam_role(auth, gke_sa_email, 'roles/artifactregistry.reader')
    return {'registry': registry_name, 'sa': gke_sa_email, 'role': 'roles/artifactregistry.reader'}